In [2]:
import os
from elasticsearch import Elasticsearch
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

es = Elasticsearch(os.getenv("ES_HOST"), basic_auth=(os.getenv("ES_USER"), os.getenv("ES_PASSWORD")))

# Check connection
if es.ping():
    print("Connected to Elasticsearch")
else:
    print("Failed to connect to Elasticsearch")

Connected to Elasticsearch


In [4]:
response = es.search(
    index="gumball_transcripts",
    query={"match_all": {}},
    size=10000
)

documents = response["hits"]["hits"]

print(f"Loaded {len(documents)} documents")

Loaded 256 documents


In [ ]:
from elasticsearch.helpers import bulk

# Calculate word counts for each document
word_counts = []
for doc in documents:
    text = doc["_source"]["text"]
    word_count = len(text.split())
    word_counts.append(word_count)

# Sort word counts and calculate percentiles
sorted_word_counts = sorted(word_counts)
p33 = sorted_word_counts[int(0.33 * len(sorted_word_counts))]
p66 = sorted_word_counts[int(0.66 * len(sorted_word_counts))]

# Prepare bulk update actions
actions = []
for doc, wc in zip(documents, word_counts):
    if wc < p33:
        length_category = "short"
    elif wc < p66:
        length_category = "medium"
    else:
        length_category = "long"
    
    # Add the length field to the document
    doc["_source"]["length"] = length_category

    # Prepare the update action
    action = {
        "_op_type": "update",
        "_index": "gumball_transcripts",
        "_id": doc["_id"],
        "doc": {"length": length_category}
    }
    actions.append(action)

# Perform bulk update
bulk(es, actions)

print("Updated documents with length categories.")

Length	short	Word Count	2496	ID	usXQrZQBnq1yqBJ4H4gy
Length	medium	Word Count	2797	ID	u8XQrZQBnq1yqBJ4H4ho
Length	long	Word Count	3099	ID	vMXQrZQBnq1yqBJ4H4iI
Length	short	Word Count	2514	ID	vcXQrZQBnq1yqBJ4H4ip
Length	short	Word Count	2543	ID	vsXQrZQBnq1yqBJ4H4jJ
Length	long	Word Count	3366	ID	v8XQrZQBnq1yqBJ4H4jo
Length	long	Word Count	3193	ID	wMXQrZQBnq1yqBJ4IIgP
Length	short	Word Count	286	ID	wcXQrZQBnq1yqBJ4IIgu
Length	medium	Word Count	2656	ID	wsXQrZQBnq1yqBJ4IIhK
Length	long	Word Count	2960	ID	w8XQrZQBnq1yqBJ4IIhn
Length	long	Word Count	2902	ID	xMXQrZQBnq1yqBJ4IIiE
Length	medium	Word Count	2780	ID	xcXQrZQBnq1yqBJ4IIin
Length	medium	Word Count	2763	ID	xsXQrZQBnq1yqBJ4IIjH
Length	long	Word Count	3250	ID	x8XQrZQBnq1yqBJ4IIjp
Length	medium	Word Count	2628	ID	yMXQrZQBnq1yqBJ4IYgM
Length	short	Word Count	2441	ID	ycXQrZQBnq1yqBJ4IYgt
Length	medium	Word Count	2660	ID	ysXQrZQBnq1yqBJ4IYha
Length	long	Word Count	3161	ID	y8XQrZQBnq1yqBJ4IYh6
Length	long	Word Count	2840	ID	zMXQrZQBnq1yqBJ4I